# Trump Tweet Fine‑Tuning (GPT‑2 Large + LoRA)
Wykorzystuje LoRA (Low‑Rank Adaptation), aby szybko dostroić duży model (GPT‑2 Large 774M) do stylu Trumpa.
Czas treningu < 15 minut przy 10k tweetach i 2 epokach.

In [ ]:
# Instalacja potrzebnych bibliotek (bez Colab)
!pip install -q torch transformers datasets pandas kaggle peft accelerate
!pip install -q --upgrade torchao   # <-- dodaj to

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.2 MB/s eta 0:00:00


In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
import time

# GPU?
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Set up Kaggle API from Colab Secrets
from google.colab import userdata
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')

# Save credentials to ~/.kaggle/kaggle.json
import os
import json
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
print('Kaggle API configured')

In [ ]:
# Pobieranie datasetu Trump's Legacy
import subprocess
import zipfile

dataset_path = Path('/tmp/trump_data')
dataset_path.mkdir(exist_ok=True)

print('Pobieranie datasetu Trump\'s Legacy...')
subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'zusmani/trumps-legacy', '-p', str(dataset_path)],
    check=True
)

zip_file = list(dataset_path.glob('*.zip'))[0]
with zipfile.ZipFile(zip_file, 'r') as z:
    z.extractall(dataset_path)
print(f'Dataset rozpakowany do {dataset_path}')
print('Pliki:', list(dataset_path.glob('*.csv'))[:3])

In [ ]:
# Wczytanie tweetów
csv_files = list(dataset_path.glob('*.csv'))
df = pd.read_csv(csv_files[0])
print(f'Wczytano {len(df)} tweetów')
print(f'Kolumny: {df.columns.tolist()}')
print('\nPierwsze 3 tweety:')
print(df['text'].head(3).values)

In [ ]:
# Czyszczenie tweetów
import re

def clean_tweet(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_tweet)
df = df[df['text'].str.len() > 10]
print(f'Po czyszczeniu: {len(df)} tweetów')

In [ ]:
# Ograniczenie danych do 10k tweetów (szybki trening <15 min)
SAMPLE_SIZE = 10_000
df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f'Użyjemy {len(df)} tweetów do treningu.')

In [ ]:
# Tokenizacja
from transformers import AutoTokenizer

BASE_MODEL = 'gpt2-large'   # 774M parametrów
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

encodings = tokenizer(
    df['text'].tolist(),
    truncation=True,
    max_length=128,
    padding='max_length',
    return_tensors='pt'
)
print(f'Tokenizacja: {encodings["input_ids"].shape[0]} tweetów')

In [ ]:
# Podział na trening/walidację (80/20)
from torch.utils.data import TensorDataset, DataLoader, random_split

dataset = TensorDataset(encodings['input_ids'], encodings['attention_mask'])
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

BATCH_SIZE = 8          # mniejszy batch, żeby zmieścić model na GPU
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
print(f'Trening: {len(train_loader)} batchy, Walidacja: {len(val_loader)} batchy')

In [ ]:
# Załadowanie modelu bazowego i nałożenie LoRA
from transformers import AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(device)

lora_config = LoraConfig(
    r=8,                      # rank
    lora_alpha=16,            # skalowanie
    target_modules=["c_attn", "c_proj"],  # warstwy uwagi w GPT-2
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()  # pokaże, ile % parametrów jest trenowanych
model.train()

In [ ]:
# Konfiguracja treningu (LoRA używa wyższego LR)
from torch.optim import AdamW

NUM_EPOCHS = 2
LEARNING_RATE = 5e-4          # typowe dla LoRA
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

print(f'Epoki: {NUM_EPOCHS}, LR: {LEARNING_RATE}, Batch size: {BATCH_SIZE}')
print('Rozmiar zbioru treningowego:', len(train_dataset))

In [ ]:
# Pętla treningowa (z pomiarem czasu)
import time
from tqdm.notebook import tqdm

best_val_loss = float('inf')
checkpoint_dir = Path('/tmp/trump_lora_checkpoints')
checkpoint_dir.mkdir(exist_ok=True)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # --- Trening ---
    model.train()
    train_loss = 0
    progress = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]')
    for batch_idx, (input_ids, attention_mask) in enumerate(progress):
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_loss += loss.item()
        progress.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train_loss = train_loss / len(train_loader)

    # --- Walidacja ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for input_ids, attention_mask in val_loader:
            input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=input_ids)
            val_loss += outputs.loss.item()
    avg_val_loss = val_loss / len(val_loader)

    elapsed = (time.time() - start_time) / 60
    print(f'Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f} [czas: {elapsed:.1f} min]')

    # Zapis najlepszego adaptera LoRA
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained(checkpoint_dir / 'best_model')
        print(f'  -> Zapisano najlepszy model (val loss: {avg_val_loss:.4f})')

total_time = (time.time() - start_time) / 60
print(f'\nCałkowity czas fine‑tuningu: {total_time:.1f} minut')

In [ ]:
# Generowanie tweetów (wczytanie adaptera LoRA)
from peft import PeftModel

# Załaduj bazowy model i ponownie nałóż zapisany adapter
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(device)
model = PeftModel.from_pretrained(base_model, checkpoint_dir / 'best_model')
model.config.pad_token_id = tokenizer.eos_token_id
tokenizer.pad_token = tokenizer.eos_token
model.eval()

prompts = ['Make America', 'I will', 'The fake news', 'Beautiful', 'Total disaster']
print('Wygenerowane tweety w stylu Trumpa:\n')

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    output = model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=50,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.1,
        no_repeat_ngram_size=2
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f'Prompt: "{prompt}"')
    print(f'Generated: "{text}"\n')

In [ ]:
# Perplexity porównawcze (oryginał vs dostrojony)
import math

print('Perplexity (im niższa, tym lepiej)\n')
finetuned_ppl = math.exp(best_val_loss)
print(f'Dostrojony model: {finetuned_ppl:.2f}')

# Oryginalny GPT-2 Large
orig_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(device)
orig_model.eval()
orig_val_loss = 0
with torch.no_grad():
    for input_ids, attention_mask in val_loader:
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        outputs = orig_model(input_ids, attention_mask=attention_mask, labels=input_ids)
        orig_val_loss += outputs.loss.item()
orig_val_loss /= len(val_loader)
orig_ppl = math.exp(orig_val_loss)
print(f'Oryginalny GPT-2 Large: {orig_ppl:.2f}')
print(f'\nPoprawa: {orig_ppl - finetuned_ppl:.2f} punktów ({(orig_ppl/finetuned_ppl - 1)*100:.1f}% redukcji)')